# Ingest marine embeddings (Free Edition–friendly)

Playground demo path (no Flask dashboard):

1. Run `notebooks/spark_marine_pipeline.py` first (sync docs into Lakebase)
2. Run this notebook (embeddings)
3. Use Playground + MCP tools (`search_marine_context`, `assess_and_act`, …)

Notes:

- Uninstall conflicting `psycopg2` / `psycopg2-binary`, then `%pip` install deps + `restartPython`
- HuggingFace caches under `/tmp`
- Embed with `sentence-transformers`
- Write to Lakebase with **psycopg2** (no Spark JDBC for vectors)

**Run cells one at a time.** After restart, wait until the kernel shows Connected.

In [ ]:
%pip uninstall -y psycopg2 psycopg2-binary
%pip install -q 'databricks-sdk>=0.118.0' sentence-transformers requests

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os
from pathlib import Path

os.environ["HF_HOME"] = "/tmp/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/tmp/.cache/huggingface"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
Path("/tmp/.cache/huggingface").mkdir(parents=True, exist_ok=True)

print("Config ready")

In [ ]:
import importlib.util
from pathlib import Path

script = None
roots = [Path.cwd(), Path.cwd().parent, *list(Path.cwd().parents)[:5]]
for root in roots:
    for hit in (
        root / "notebooks" / "ingest_marine_embeddings.py",
        root / "ingest_marine_embeddings.py",
    ):
        if hit.exists():
            script = hit
            break
    if script is not None:
        break

if script is None:
    raise FileNotFoundError("Could not find ingest_marine_embeddings.py")

print("Loading", script)
spec = importlib.util.spec_from_file_location("ingest_marine_embeddings", script)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()